In [ ]:
"""
Given a channel type, conduct certain number of SOBOL + BO trials.
Within each trial, conduct n_trials of simulations with varied friction
coefficients to get an averaged speed performance.
"""
import sys
print(sys.executable)
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig, ChoiceParameterConfig
from ax.utils.notebook.plotting import init_notebook_plotting
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationStep
from ax.generation_strategy.transition_criterion import MaxTrials
from ax.adapter.registry import Generators
import numpy as np
from shapely.wkt import loads
from bo_sheet.sim_wrapper_BO import shape_objective_function
import os
import time

material_dict = {}
fixed_parameters = {}

# [density (g/mm^3), Young's modulus (MPa), magnetization density (A/m)]
material_dict['13_9_2025_pdms_1_1'] = [1.795 * 1e-3, 2.2, 187298.43770657358]

fixed_parameters['robot_width'] = 2.0 # [mm]

fixed_parameters['target_dl'] = 0.25 # [mm]
fixed_parameters['cfl'] = 0.4

fixed_parameters['B_field'] = 0.05 # [T]
fixed_parameters['B_frequency'] = 4.0 # [Hz]

fixed_parameters['backward'] = False

fixed_parameters['final_time'] = 20e3 # [ms]

fixed_parameters['save_directory'] = os.path.join(os.path.expanduser('~'), 'Documents', 'GitHub', 'Bayesian_Optimization_for_Sheet_Robots', \
                              'code', '3_optim_shape')

fixed_parameters['channel_type'] = 'coronary_artery'
fixed_parameters['patient_name'] = 'zeroFour'
fixed_parameters['image_name'] = 'zeroFour_right'
fixed_parameters['trial_num'] = 'trial_0'
fixed_parameters['segment_str_list'] = ['max_curvature', 'max_width', 'min_width', 'max_curvature_change', 'max_width_change']

fixed_parameters['variation_level'] = 0.5

# Varying friction coefficients at every contact step around the optimal friction coefficients
fixed_parameters['dynamic_fric_coeff'] = True
fixed_parameters['dynamic_fric_variation_level'] = 0.2

N_sobol = 24
N_BO = 30
batch_size = 1

# Number of channels to be repeatedly constructed
fixed_parameters['n_channels'] = 1

# Number of varied friction cases to be done for a channel
fixed_parameters['n_trials'] = 4

fixed_parameters['upper_bound_robot_length'] = 10.0

fixed_parameters['static_mu_wall'] = 8.937293906750346
fixed_parameters['kinetic_mu_wall'] = 3.7862437848024757
fixed_parameters['static_mu_wall_end'] = 13.249480260330722
fixed_parameters['kinetic_mu_wall_end'] = 1.7875154798884043
fixed_parameters['static_mu_substrate'] = 0.12889650832825794
fixed_parameters['kinetic_mu_substrate'] = 0.017422378787529738
fixed_parameters['k_mag_den'] = 1.538629490914029
fixed_parameters['k_modulus'] = 1.1194380619912316
fixed_parameters['end_line_percent'] = 1.0
fixed_parameters['precise_offset'] = 0.2 # mm for the left robot tip to be away from boundary
# to avoid strong propulsion force at time = 0

# ==================================================================================================================================================================

channel_polygon_list = []
x_midline_list = []
y_midline_list = []
left_bank_list = []
right_bank_list = []
total_length_list = []

for i in range(len(fixed_parameters['segment_str_list'])):
    with open(os.path.join(fixed_parameters['save_directory'], 'channel_params', fixed_parameters['channel_type'],\
                           fixed_parameters['patient_name'], fixed_parameters['trial_num'],\
                            f"{fixed_parameters['image_name']}_{fixed_parameters['segment_str_list'][i]}_polygon.wkt"), 'r') as file:
        channel_polygon = loads(file.read())
        file.close()

    with np.load(os.path.join(fixed_parameters['save_directory'], 'channel_params', fixed_parameters['channel_type'],\
                           fixed_parameters['patient_name'], fixed_parameters['trial_num'],\
                            f"{fixed_parameters['image_name']}_{fixed_parameters['segment_str_list'][i]}_channel_info.npz")) as channel_info:
        x_midline = channel_info['x_midline']
        y_midline = channel_info['y_midline']
        left_bank = channel_info['left_bank']
        right_bank = channel_info['right_bank']
        total_length = channel_info['total_length']
        channel_info.close()

    channel_polygon_list.append(channel_polygon)
    x_midline_list.append(x_midline)
    y_midline_list.append(y_midline)
    left_bank_list.append(left_bank)
    right_bank_list.append(right_bank)
    total_length_list.append(total_length)

fixed_parameters['channel_polygon_list'] = channel_polygon_list
fixed_parameters['x_midline_list'] = x_midline_list
fixed_parameters['y_midline_list'] = y_midline_list
fixed_parameters['left_bank_list'] = left_bank_list
fixed_parameters['right_bank_list'] = right_bank_list
fixed_parameters['total_length_list'] = total_length_list
fixed_parameters['offset_factor'] = 0.0

del channel_polygon, x_midline, y_midline, left_bank, right_bank, total_length, channel_polygon_list, x_midline_list, y_midline_list, left_bank_list, right_bank_list, total_length_list

client = Client()

# Minimal robot length > 3 * maximum channel width
# Maximum robot thickness < 0.8 * minimum channel width
parameters = [
    RangeParameterConfig(
        name='robot_length', bounds=(5.0, fixed_parameters['upper_bound_robot_length']), parameter_type='float'
    ),
    RangeParameterConfig(
        name='robot_thickness', bounds=(0.1, 0.3), parameter_type='float'
    ),
    RangeParameterConfig(
        name='robot_n_waveform', bounds=(0.5, 2.5), parameter_type='float'
    ),
    ChoiceParameterConfig(
        name='robot_material', values=['13_9_2025_pdms_1_1'], parameter_type='str'
    )
]

client.configure_experiment(parameters=parameters)
client.configure_optimization(objective="-summedTime")

gs = GenerationStrategy(
    steps=[
        GenerationStep(
            generator=Generators.SOBOL,
            num_trials=N_sobol,
            min_trials_observed=0,
            max_parallelism=batch_size,
            completion_criteria=[MaxTrials(N_sobol)]
        ),
        GenerationStep(
            generator=Generators.BOTORCH_MODULAR,
            num_trials=N_BO,
            min_trials_observed=0,
            max_parallelism=batch_size
        )
    ]
)

client.set_generation_strategy(gs)
init_notebook_plotting()

success_counter = 0
start_time = time.time()

while success_counter < (N_sobol + N_BO):
    trials = client.get_next_trials(max_trials=batch_size)

    for trial_index, BO_parameters in trials.items():

        if BO_parameters['robot_length'] < BO_parameters['robot_n_waveform'] * (2.0 * np.pi * BO_parameters['robot_thickness'] + 0.1):
            trial_status = client.mark_trial_failed(trial_index=trial_index, failed_reason="Violated parameter constraint")
            continue

        fixed_parameters['robot_density'], fixed_parameters['robot_modulus'], fixed_parameters['robot_magnetization_density'] = material_dict[BO_parameters['robot_material']]

        objectives = shape_objective_function(BO_parameters, fixed_parameters)

        with open(os.path.join(fixed_parameters['save_directory'], 'channel_results', fixed_parameters['channel_type'],\
                               fixed_parameters['patient_name'], fixed_parameters['trial_num'],\
                               'debug.txt'), 'a') as file:
            file.write(f"Robot length: {BO_parameters['robot_length']}\n")
            file.write(f"Robot thickness: {BO_parameters['robot_thickness']}\n")
            file.write(f"Robot n waveform: {BO_parameters['robot_n_waveform']}\n")
            file.write(f"Robot material: {BO_parameters['robot_material']}\n")
            file.write(f"Trial {trial_index}. Jupyter: {objectives}\n")
            file.close()

        trial_status = client.complete_trial(trial_index=trial_index, raw_data=objectives)
        client.save_to_json_file(os.path.join(fixed_parameters['save_directory'], 'channel_results', fixed_parameters['channel_type'],\
                                                fixed_parameters['patient_name'], fixed_parameters['trial_num'],\
                                                f"{success_counter}.json"))
        success_counter += 1

best_parameters, best_metrics, best_trial_index, _ = client.get_best_parameterization(False)

with open(os.path.join(fixed_parameters['save_directory'], 'channel_results', fixed_parameters['channel_type'],\
                        fixed_parameters['patient_name'], fixed_parameters['trial_num'],\
                        'result.txt'), 'a') as file:
    file.write("From the raw best simulated result.\n")
    file.write("Best parameters:\n")
    file.write(f"{best_parameters}\n")
    file.write("Best metrics (Mean, variance):\n")
    file.write(f"{best_metrics}\n")
    file.write(f"Corresponding trial index: {best_trial_index}\n")
    file.write("\n")
    file.write(f"Friction variation level: {fixed_parameters['variation_level']}\n")
    file.write(f"Number of SOBOL: {N_sobol}\n")
    file.write(f"Number of BO steps: {N_BO}\n")
    file.write("---------------------------------------------------------------------------\n\n")
    file.close()

best_parameters, best_metrics, best_trial_index, _ = client.get_best_parameterization()

# If you do crash, crash at least after having a save.
delta_time = time.time() - start_time

with open(os.path.join(fixed_parameters['save_directory'], 'channel_results', fixed_parameters['channel_type'],\
                        fixed_parameters['patient_name'], fixed_parameters['trial_num'],\
                        'result.txt'), 'a') as file:
    file.write("From SingleTaskGP (default BayesOpt setup by Ax) prediction.\n")
    file.write("Best parameters:\n")
    file.write(f"{best_parameters}\n")
    file.write("Best metrics (Mean, variance):\n")
    file.write(f"{best_metrics}\n")
    file.write(f"Corresponding trial index: {best_trial_index}\n")
    file.write("\n")
    file.write(f"Friction variation level: {fixed_parameters['variation_level']}\n")
    file.write(f"Number of SOBOL: {N_sobol}\n")
    file.write(f"Number of BO steps: {N_BO}\n")
    file.write(f"Time taken to execute all BO: {delta_time}s\n\n")
    file.write("---------------------------------------------------------------------------\n\n")
    file.close()

print(f"Log appended to {os.path.join(fixed_parameters['save_directory'], 'channel_results', fixed_parameters['channel_type'], fixed_parameters['patient_name'], fixed_parameters['trial_num'], 'result.txt')}")

analysis_general = client.compute_analyses()